In [1]:
import pandas as pd
from query_gym.dataset.unite_dataset import UniteDataset
import ipywidgets
from ipywidgets import interact, Dropdown, Button, VBox, Output, HBox, Text, Textarea, Layout, IntProgress
from rich.console import Console
from rich.table import Table
from query_gym.envs.query_gym_sqlite import SQLiteQueryGym
from functools import partial
import time, threading
from IPython.display import display, Markdown
from IPython.display import HTML
import time
import json
import ast     
from pandas import read_sql_query
   
train_dataset = UniteDataset(
    dataset_path="/data/datasets/unified/bird_dev",
    mode="dev",
)
with open("./demo_examples.jsonl", "r") as f:
    EXAMPLES = [json.loads(x) for x in f]
env = SQLiteQueryGym(dataset=train_dataset)
ex_id = 820
temp_table = env.reset(ex_id)
ex = EXAMPLES[0]
trajectory = ex['qpg_trajectory']
def on_example_change(change):
    global ex_id
    global example
    global ex
    if change['type'] == 'change' and change['name'] == 'value':
        ex_id = EXAMPLES[int(change['new'].split(".")[0])-1]['example_id']
        with example_output:
            example_output.clear_output()
            print(f"Natural Language Query: {change['new']}")
            print("-"*30)
            print("Initial Observation: ")
            print("-"*30)
            print(env.reset(ex_id))
        example = [val for val in EXAMPLES if val['example_id'] == ex_id]
        ex = example[0]


def on_table_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        table = change['new']
        with schema_output:   
            schema_output.clear_output()
            if table != "Select Table":
                print("Schema of the table:")
                print(read_sql_query(f"SELECT  sql FROM sqlite_master WHERE type='table' and name = '{table}'; ", env.conn)['sql'][0])
                print("Preview of the table looks like:")
                display(env.cte[table].head(3))

def execute_action(action, args=None):
    console.rule(f"[yellow]{action}[/yellow]")
    try:
        parts = ast.literal_eval(args)
        o,r,d,t,i = env.step(parts)
        console.print(o)
    except Exception as e:
        console.print(f"[red]Invalid args format:[/red] {e}")
        return

def on_click(b):
    output.clear_output()
    with output:
        action = action_dropdown.value
        args = command_box.value
        execute_action(action, args)

def query_getter(sql_type, sql_box):
    # Fake LLM delay
    if sql_type == 'vanilla':
        sql_box.value = "Generating...."
        time.sleep(2)
        return ex['vanilla_sql']
    else:
        return ex['ground_truth_sql']
        

def on_action_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected = change['new']
        value = ACTION_USAGE.get(selected, "")
        usage_box.value = f"{value['usage']}"
        if value["example"]:
            command_box.value = value["example"]
        else:
            command_box.value = value["usage"]

def make_sql_section(label, button_label, color, sql_type):
    global ex
    lp_failure=lambda: ex['logical_plan_failure']
    sql_box = Textarea(
        placeholder=f"{label} SQL will appear here",
        description=f"{label}:",
        layout={'width': '90%', 'height': '80px'}
    )
    explain_btn = Button(description="Explain")
    gen_btn = Button(description=button_label, button_style=color)
    convert_btn = Button(description=f"Convert to env steps", button_style='info')
    steps_box = VBox() 
    sql_output = Output()
    explanation = Output()
    progress = IntProgress(value=0, min=0, max=100, description='Generating:')

    def on_explain(b):
        with explanation:
            print(ex['vanilla_explanation'])

    def on_generate(b):
        with nl_output:
            nl_output.clear_output()
            print("PROMPT:")
            print("NL Question:", ex['nl_question'])
            print("Generate SQL: {GET_OVERVIEW}, {User Query} and any associated {Evidence}")

        # sql_box.value = "Generating..."    
        def run():
            result = query_getter(sql_type, sql_box)
            sql_box.value = result
            env.prefix_tracker['T'] = 0
        threading.Thread(target=run).start()
    

    def run_step(step, out, b):
        from IPython.display import display, HTML
        bad_color = "#ffe6e6"
        good_color = "#eaffea"
        with out:
            out.clear_output()
            try:
                o, r, d, t, i = env.step(step)
                if i['operation_success']:
                    color = good_color
                else:
                    color = bad_color
                display(HTML(f"""
                    <div style='background:#e6f0ff; border:1px solid #3399ff; 
                                padding:8px; border-radius:6px; margin:4px 0;'>
                        <b>Step executed:</b>
                        <pre style="margin:0; white-space:pre-wrap;">{step}</pre>
                    </div>
                """))
    
                # Result
                display(HTML(f"""
                    <div style='background:{color}; border:1px solid {color}; 
                                padding:8px; border-radius:6px; margin:4px 0;'>
                        <b>Env Observation:</b>
                        <pre style="margin:0; white-space:pre-wrap;">{o}</pre>
                    </div>
                """))
    
            except Exception as e:
                display(HTML(f"""
                    <div style='background:#ffe6e6; border:1px solid #ff3333; 
                                padding:8px; border-radius:6px; margin:4px 0;'>
                        <b>FAILED:</b>
                        <pre style="margin:0; white-space:pre-wrap;">\n\n{e}</pre>
                    </div>
                """))


    def on_convert(b):
        steps = lp_failure()
        step_widgets = []

        for i, step in enumerate(steps):
            if isinstance(step, list):
                display_text = "\n".join(str(x) for x in step)
            else:
                display_text = str(step)
            step_out = Output()
            exec_btn = Button(description=f"Run Step {i+1}", button_style='primary')
            exec_btn.on_click(partial(run_step, step, step_out))

            step_widgets.append(
                VBox([
                    Textarea(value=display_text,
                             layout={'width': '90%', 'height': '100px'}),
                    exec_btn,
                    step_out
                ])
            )
        steps_box.children = step_widgets
    gen_btn.on_click(on_generate)
    convert_btn.on_click(on_convert)
    explain_btn.on_click(on_explain)


    return VBox([HBox([gen_btn, explain_btn]), sql_box, explanation, convert_btn, steps_box, sql_output])


def render_step(i):
    global trajectory
    try: 
        step = trajectory[i]
    except Exception as e:
        pass
    with step_output:
        step_output.clear_output()
        step_out, result_out, render_op = format_step(step)
        display(step_out)

def on_agent_start(b):
    idx["value"] = 0
    with agent_output:
        agent_output.clear_output()
        print("Starting Single Agent Demo")
    with log_output:
        log_output.clear_output()
    render_step(0)

def on_play(b):
    global ex
    trajectory = ex["qpg_trajectory"] 
    with log_output:
        log_output.clear_output()
        for i, step in enumerate(trajectory):
            step_out, result_out, result_op = format_step(step)
            display(step_out)
        idx["value"] = len(trajectory) - 1
        render_step(idx["value"])

def format_step(step):
    role = step['role'].lower()
    content = step['content']
    reward = step['reward']
    done_value = step['reward']==100
    goal_color = 'lightpink'
    op_color = 'lightpink'
    if step['operation_success']:
        op_color = 'lightgreen'
    if done_value:
        goal_color = 'lightgreen'
        
    colors = {
        "agent": "lavender",
        "system": "moccasin",
        "environment": "lightblue"
    }

    color = colors.get(role, "white")  # fallback to white
    label = role.capitalize()

    return (HTML(f"""
        <div style="background-color:{color}; padding:6px; border-radius:6px; margin:4px 0;">
            <b>{label.upper()}:</b>
            <code style="white-space:pre-wrap; display:block;">{content}</code>
        </div>
    """), HTML(f"""
        <div style="background-color:{goal_color}; padding:6px; border-radius:6px; margin:4px 0;">
            <b>GOAL ACHIEVED: {done_value}</b>
        </div>
    """),
    HTML(f"""
        <div style="background-color:{op_color}; padding:6px; border-radius:6px; margin:4px 0;">
            <b>OPERATION EXECUTED: {step['operation_success']}</b>
        </div>
    """))

def trajectory_explorer_vertical(trajectory):
    output = Output(layout=Layout(width="70%", height="400px"))
    role_colors = {
        "system": "lightyellow",
        "human": "lightblue",
        "agent": "plum",
        "environment": "lightgrey"
    }
    role_borders = {
        "true": "green",
        "false": "red",
        "none": "black",
    }

    def make_onclick(step):
        def on_click(b):
            with output:
                output.clear_output()
                op1, op2, op3 = format_step(step)
                display(op1)
                display(op2)
                if step['operation_success'] is not None:
                    display(op3)
        return on_click

    column_items = []
    for i, step in enumerate(trajectory):
        bcolor = role_borders.get(str(step['operation_success']).lower(), "black")
        btn = Button(
            description=f"{step['description']}",
            tooltip=step['content'],
            layout=Layout(
                width="200px",
                border=f"2px solid {bcolor}",
                border_radius="6px",
                margin="4px auto"
            ),
            style=dict(
                button_color=role_colors.get(step['role'].lower(), "white")
            )
        )
        btn.on_click(make_onclick(step))
        column_items.append(btn)

        if i < len(trajectory) - 1:
            arrow = ipywidgets.HTML(value="<div style='font-size:20px;'>&#8595;</div>")
            column_items.append(arrow)

    return HBox([
        VBox(column_items, layout=Layout(align_items="center", width="30%")),
        output
    ])
    
ACTION_USAGE = {
    "SELECT ACTION": {"usage": "", "example": ""},
    "GET_OVERVIEW": {"usage": '["get_overview"]', "example": ""},
    "GET_SCHEMA": {"usage": '["get_schema"]', "example": ""},
    "GET_QUERY": {"usage": '["get_query"]', "example": ""},
    "GET_TABLES": {"usage": '["get_tables"]',  "example": ""},
    "GET_COLUMNS": {"usage": '["get_columns", "<table_id>"]',  "example": "['get_columns', 'superhero']"},
    "GET_ACTIONS": {"usage": '["get_actions"]', "example": ""},
    "GET_OPERATIONS": {"usage": '["get_operations"]', "example": ""},
    "PREVIEW_TABLE": {"usage": '["preview_table", "<table_id>"]', "example": '["preview_table", "attribute"]'},
    "GET_COLUMN_STATS": {"usage": '["get_column_stats", "<table_id>", "<column_id>"]', "example": '["get_column_stats", "hero_attribute", "attribute_value"]'},
    "GET_UNIQUE_VALUES": {"usage": '["get_unique_values", "<table_id>", "<column_id>"]', "example": '["get_unique_values", "superpower", "power_name"]'},
    "GET_COLUMN_TYPES": {"usage":'["get_column_types", "<table_id>"]', "example": '["get_column_types", "alignment"]'},
    "GET_SAMPLE_VALUES": {"usage":'["get_sample_values", "<table_id>", "<column_id>"]', "example": '["get_sample_values", "superhero", "superhero_name"]'},
    "PERFORM_PROJECTION": {"usage": '["perform_projection", "<table_id>", "<columns>"]', "example": '["perform_projection", "superhero", "superhero_name, full_name"]'},
    "PERFORM_FILTER": {"usage": '["perform_filter", "<table_id>", "<conditions>", "[projected columns]"]', "example": '["perform_filter", "superhero", "superhero_name=\'3-D Man\'", "full_name"]'},
    "PERFORM_JOIN": {"usage": '["perform_join", "<list-of-tables>", "<list-of-join-conditions>", "<list-of-join-type>", "<projected columns>"]', "example": '["perform_join", ["superhero AS T1", "hero_attribute AS T2", "attribute AS T3"], ["T1.id = T2.hero_id", "T3.id = T2.attribute_id"], ["INNER JOIN", "INNER JOIN"]]'},
    "PERFORM_ORDER_BY": {"usage": '["perform_order_by", "<table-id>", "<order-condition>", "[projected columns]"]', "example": '["perform_order_by", "superhero", "weight_kg DESC", "superhero.full_name, superhero.superhero_name, superhero.weight_kg"]'},
    "PERFORM_LIMIT": {"usage": '["perform_limit", "<table-id>", "<integer>", "[projected columns]"]', "example": '["perform_limit", "superhero", "10"]'},
    "PERFORM_AGGREGATE": {"example": '["perform_aggregate", "superhero", "gender_id", "AVG(height_cm)", "skin_colour_id = 1"]', "usage": '["perform_aggregate", "<table_id>", "<column_id>", "<projected_columns>", "[having_condition]"]'},
    "PERFORM_UNION": {"usage": '["perform_union", "ALL|DISTINCT", "<table-1-id>", "<table-2-id>", "[table-1-columns]", "[table-2-columns]"]', "example": ""},
    "PERFORM_INTERSECT": {"usage": '["perform_intersect", "<table-1-id>", "<table-2-id>", "[table-1-columns]", "[table-2-columns]"]',"example": ""}
}

In [2]:
from IPython.display import display, HTML, Image
display(HTML("<div style='text-align:center;'><img src='images/authors.png' width='600'></div>"))
display(HTML("<div style='text-align:center;'><img src='images/qpgym.png' width='600'></div>"))

In [3]:
from IPython.display import display, HTML, Image

display(HTML("<div style='text-align:center;'><img src='images/flowdiagram.png' width='500'></div>"))


## Choose Example

In [ ]:
options = ["Choose Example"]
options.extend([f"{i+1}. {data['nl_question']}" for i,data in enumerate(EXAMPLES)])
example_dropdown = Dropdown(
    options=options,
    description="Example:"
)
example_output = Output()
example_dropdown.observe(on_example_change, names="value")
VBox([example_dropdown, example_output])

## Schema Explorer - Explore all the tables/columns in the selected database. 

### Choose tables to preview.

In [5]:
tables = list(env.cte.keys())
tables.insert(0,'Select Table')
table_dropdown = Dropdown(options=tables, description="Table:")
schema_output = Output()
table_dropdown.observe(on_table_change, names='value')
display(VBox([table_dropdown, schema_output]))

## Action Explorer - Explore the action space to see how an LLM based agent would interact with the environment.

In [6]:
action_dropdown = Dropdown(options=list(ACTION_USAGE.keys()), description="Action:")
run_btn = Button(description="Run Action", button_style="success")
command_box = Text(
    value='Perform Operation Args',
    description="Execute:",
    layout={'width': '80%'}
)

usage_box = Text(
    value='Usage will appear here',
    description="Usage:",
    layout={'width': '80%'}
)

usage_output = Output()
output = Output()
console = Console()
action_dropdown.observe(on_action_change)
run_btn.on_click(on_click)
VBox([HBox([action_dropdown, run_btn]), usage_box, command_box, output])

## Single shot Text2SQL Generation - Drawbacks

In [ ]:
nl_output = Output()
vanilla_section = make_sql_section(
    label="Vanilla",
    button_label="Generate Vanilla SQL",
    color="danger",
    sql_type='vanilla',
)

display(VBox([nl_output, vanilla_section]))

## Single Agent Demo - Interactive View of how an agent can plan and remediate errors.

In [ ]:
from IPython.display import display
from ipywidgets import Button
vizoutput = Output()
def on_visualize(b):
    with vizoutput:
        vizoutput.clear_output()
        global ex
        trajectory = ex["qpg_trajectory"] 
        viz = trajectory_explorer_vertical(trajectory)
        display(viz) 

start_single_agent_button = Button(
    description="Single Agent", 
    button_style="primary"
)
start_single_agent_button.on_click(on_visualize)
display(start_single_agent_button)
display(vizoutput)

# Visualizing the flow of conversation

In [ ]:
step_output = Output()       
log_output = Output()        
play_btn = Button(description="Play Single Agent", button_style="primary")
idx = {"value": 0}
play_btn.on_click(on_play)
demo_ui = VBox([
    play_btn,
    log_output
])
display(demo_ui)